# Feature Engineering — credit_card_balance.csv

**¿Qué es esta tabla?**
Contiene el estado mensual de las tarjetas de crédito anteriores de cada cliente.
Con 3.8M de filas es la tabla más pequeña de las secundarias.

**¿Qué captura?**
Comportamiento de tarjeta: saldo utilizado, límite disponible, pagos mínimos,
cuánto del límite está ocupado. El ratio de utilización de tarjeta (saldo/límite)
es uno de los predictores de riesgo más usados en la industria crediticia.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/raw/'
print('Librerías cargadas.')

Librerías cargadas.


## 1. Exploración

In [2]:
cc = pd.read_csv(DATA_PATH + 'credit_card_balance.csv')
print(f'Shape: {cc.shape}')
print(f'Columnas: {cc.columns.tolist()}')
print(f'Clientes únicos: {cc["SK_ID_CURR"].nunique():,}')
print(f'Registros por cliente (promedio): {len(cc) / cc["SK_ID_CURR"].nunique():.1f}')
print(f'\nMissings >5%:')
miss = cc.isnull().mean()
print(miss[miss > 0.05].round(3).sort_values(ascending=False))

Shape: (3840312, 23)
Columnas: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE', 'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']
Clientes únicos: 103,558
Registros por cliente (promedio): 37.1

Missings >5%:
AMT_PAYMENT_CURRENT           0.200
AMT_DRAWINGS_ATM_CURRENT      0.195
AMT_DRAWINGS_OTHER_CURRENT    0.195
AMT_DRAWINGS_POS_CURRENT      0.195
CNT_DRAWINGS_ATM_CURRENT      0.195
CNT_DRAWINGS_OTHER_CURRENT    0.195
CNT_DRAWINGS_POS_CURRENT      0.195
AMT_INST_MIN_REGULARITY       0.079
CNT_INSTALMENT_MATURE_CUM     0.079
dtype: float64


## 2. Feature Engineering

In [3]:
# Ratio de utilización: qué porcentaje del límite está usado
# Alta utilización = cliente al límite de su capacidad de crédito
cc['CC_UTILIZATION'] = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1)

# Ratio de pago mínimo vs total adeudado
cc['CC_PAYMENT_MIN_RATIO'] = cc['AMT_PAYMENT_CURRENT'] / (cc['AMT_INST_MIN_REGULARITY'] + 1)

# Flag mora
cc['CC_IN_DPD'] = (cc['SK_DPD'] > 0).astype(int)

cc_agg = cc.groupby('SK_ID_CURR').agg(
    CC_COUNT=('SK_ID_PREV', 'count'),
    CC_MONTHS_COUNT=('MONTHS_BALANCE', 'count'),

    # Saldo y límite
    CC_AMT_BALANCE_MEAN=('AMT_BALANCE', 'mean'),
    CC_AMT_BALANCE_MAX=('AMT_BALANCE', 'max'),
    CC_AMT_CREDIT_LIMIT_MEAN=('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),

    # Utilización
    CC_UTILIZATION_MEAN=('CC_UTILIZATION', 'mean'),
    CC_UTILIZATION_MAX=('CC_UTILIZATION', 'max'),

    # Pagos
    CC_AMT_PAYMENT_MEAN=('AMT_PAYMENT_CURRENT', 'mean'),
    CC_AMT_PAYMENT_TOTAL=('AMT_PAYMENT_TOTAL_CURRENT', 'mean'),
    CC_PAYMENT_MIN_RATIO_MEAN=('CC_PAYMENT_MIN_RATIO', 'mean'),
    CC_PAYMENT_MIN_RATIO_MIN=('CC_PAYMENT_MIN_RATIO', 'min'),

    # Mora
    CC_SK_DPD_MEAN=('SK_DPD', 'mean'),
    CC_SK_DPD_MAX=('SK_DPD', 'max'),
    CC_IN_DPD_COUNT=('CC_IN_DPD', 'sum'),

    # Dibujos (retiros de efectivo con tarjeta)
    CC_AMT_DRAWINGS_ATM_MEAN=('AMT_DRAWINGS_ATM_CURRENT', 'mean'),
    CC_CNT_DRAWINGS_ATM_MEAN=('CNT_DRAWINGS_ATM_CURRENT', 'mean'),
).reset_index()

cc_agg['CC_DPD_RATIO'] = cc_agg['CC_IN_DPD_COUNT'] / (cc_agg['CC_MONTHS_COUNT'] + 1)

print(f'Shape cc_agg: {cc_agg.shape}')
print(f'Features generadas: {cc_agg.shape[1] - 1}')
cc_agg.head(3)

Shape cc_agg: (103558, 18)
Features generadas: 17


,SK_ID_CURR,CC_COUNT,CC_MONTHS_COUNT,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_CREDIT_LIMIT_MEAN,CC_UTILIZATION_MEAN,CC_UTILIZATION_MAX,CC_AMT_PAYMENT_MEAN,CC_AMT_PAYMENT_TOTAL,CC_PAYMENT_MIN_RATIO_MEAN,CC_PAYMENT_MIN_RATIO_MIN,CC_SK_DPD_MEAN,CC_SK_DPD_MAX,CC_IN_DPD_COUNT,CC_AMT_DRAWINGS_ATM_MEAN,CC_CNT_DRAWINGS_ATM_MEAN,CC_DPD_RATIO
0,100006,6,6,0.000000,0.00,270000.000000,0.000000,0.000000,NaN,0.000000,NaN,NaN,0.000000,0,0,NaN,NaN,0.000000
1,100011,74,74,54482.111149,189000.00,164189.189189,0.302677,1.049994,4843.064189,4520.067568,309.219990,0.999889,0.000000,0,0,2432.432432,0.054054,0.000000
2,100013,96,96,18159.919219,161420.22,131718.750000,0.115300,1.024884,7168.346250,6817.172344,266.435519,0.000000,0.010417,1,1,6350.000000,0.255556,0.010309


## 3. Guardar y validar

In [4]:
cc_agg.to_parquet('../data/processed/credit_card_features.parquet', index=False)
print('Guardado en data/processed/credit_card_features.parquet')

app = pd.read_csv(DATA_PATH + 'application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
val = app.merge(cc_agg, on='SK_ID_CURR', how='left')

print(f'\nClientes SIN historial en credit_card: {val["CC_COUNT"].isna().sum():,} ({val["CC_COUNT"].isna().mean()*100:.1f}%)')

print('\nCorrelación con TARGET (top features):')
corr = val.drop(columns='SK_ID_CURR').corr()['TARGET'].drop('TARGET')
print(corr.abs().sort_values(ascending=False).head(10).round(4))

Guardado en data/processed/credit_card_features.parquet

Clientes SIN historial en credit_card: 220,606 (71.7%)

Correlación con TARGET (top features):
CC_CNT_DRAWINGS_ATM_MEAN     0.1077
CC_AMT_BALANCE_MEAN          0.0872
CC_AMT_BALANCE_MAX           0.0688
CC_COUNT                     0.0605
CC_MONTHS_COUNT              0.0605
CC_AMT_DRAWINGS_ATM_MEAN     0.0599
CC_AMT_PAYMENT_TOTAL         0.0227
CC_IN_DPD_COUNT              0.0113
CC_AMT_CREDIT_LIMIT_MEAN     0.0089
CC_PAYMENT_MIN_RATIO_MEAN    0.0070
Name: TARGET, dtype: float64
